# Daily residual raw directional spatial semivariogram diagnostic, 2022-2025 July

For each July day in 2022-2025, this notebook plots one figure with two panels after removing a simple day-specific mean field:

```text
ColumnAmountO3 ~ hour fixed effects + latitude + longitude
```

The residual raw semivariogram is meant to diagnose how much of the raw spatial semivariogram was driven by mean-field/trend contamination. Each daily plot shows:

- 8 hourly residual empirical semivariogram curves in blue
- daily average over 8 hours as a thick black curve
- fitted `t=0` spatial semivariogram implied by three model estimates: Vecchia, Debiased Whittle, Corridor

At `t=0`, advection does not enter the fitted spatial semivariogram. Zero spatial lag is excluded; curves begin at the first observed grid lag.


In [ ]:
import os
import pickle
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
DATA_ROOT = Path('/Users/joonwonlee/Documents/GEMS_DATA')

YEARS = [2022, 2023, 2024, 2025]
MONTH = 7
COMMON_DAYS = list(range(1, 29))
LAT_RANGE = [-3.0, 2.0]
LON_RANGE = [121.0, 131.0]
SMOOTH = 0.5  # fitted model convention used in these estimates; exponential Matern
MEAN_MODEL = 'hour_dummy_plus_lat_lon_ols'
NORMALIZE_EMPIRICAL = False
Y_LIMIT = (0, 20)

# Same positive univariate lag grid used by the empirical semivariogram scripts.
# Zero spatial lag is deliberately excluded for pure spatial semivariograms.
LAT_LAGS = np.concatenate([[0.044, 0.132], np.arange(0.176, 2.3, 0.044 * 5)])
LON_LAGS = np.concatenate([[0.063, 0.126], np.arange(0.18, 2.0, 0.063 * 3)])
LAT_LAGS = np.round(LAT_LAGS, 3)
LON_LAGS = np.round(LON_LAGS, 3)

ESTIMATE_FILES = {
    'Vecchia_mm20': PROJECT_ROOT / 'outputs/day/july_22_23_24_25/real_vecc_july_22_23_24_25_mm20.csv',
    'Debiased_Whittle': PROJECT_ROOT / 'outputs/day/july_22_23_24_25/real_dw_july_22_23_24_25.csv',
    'Corridor_4x4_lag643': PROJECT_ROOT / 'outputs/day/july_22_23_24_25/real_july_corridor_width_4x4_lag643_all_fits.csv',
}

OUTPUT_DIR = PROJECT_ROOT / 'plots/directional_semivariograms/daily_spatial_semivariograms_residual_raw_three_model_2022_2025_052526'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUTPUT_DIR}')
print(f'Mean model: {MEAN_MODEL}')
print(f'Normalize empirical residual semivariogram: {NORMALIZE_EMPIRICAL}')
print('lat lags:', LAT_LAGS)
print('lon lags:', LON_LAGS)


In [ ]:
def parse_day_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'day' in df.columns:
        parsed = df['day'].astype(str).str.extract(r'(?P<year>\d{4})-(?P<month>\d{1,2})-(?P<day_num>\d{1,2})')
        for col in ['year', 'month', 'day_num']:
            if col not in df.columns or df[col].isna().all():
                df[col] = pd.to_numeric(parsed[col], errors='coerce').astype('Int64')
    elif {'year', 'month', 'day_idx'}.issubset(df.columns):
        df['day_num'] = pd.to_numeric(df['day_idx'], errors='coerce').astype('Int64') + 1
    return df


def load_one_estimate_file(model_name: str, path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    raw = parse_day_columns(raw)
    if 'status' in raw.columns:
        raw = raw.loc[raw['status'].astype(str).str.lower().eq('ok')].copy()

    if {'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time', 'est_advec_lat', 'est_advec_lon', 'est_nugget'}.issubset(raw.columns):
        mapped = pd.DataFrame({
            'model': model_name,
            'year': raw['year'],
            'month': raw['month'],
            'day_num': raw['day_num'],
            'day': raw['day'],
            'cov_name': raw.get('spec_name', model_name),
            'sigma': raw['est_sigmasq'],
            'range_lat': raw['est_range_lat'],
            'range_lon': raw['est_range_lon'],
            'range_time': raw['est_range_time'],
            'advec_lat': raw['est_advec_lat'],
            'advec_lon': raw['est_advec_lon'],
            'nugget': raw['est_nugget'],
            'loss': raw.get('loss', np.nan),
            'time': raw.get('total_s', raw.get('time', np.nan)),
        })
    else:
        mapped = pd.DataFrame({
            'model': model_name,
            'year': raw['year'],
            'month': raw['month'],
            'day_num': raw['day_num'],
            'day': raw['day'],
            'cov_name': raw.get('cov_name', model_name),
            'sigma': raw['sigma'],
            'range_lat': raw['range_lat'],
            'range_lon': raw['range_lon'],
            'range_time': raw['range_time'],
            'advec_lat': raw['advec_lat'],
            'advec_lon': raw['advec_lon'],
            'nugget': raw['nugget'],
            'loss': raw.get('loss', np.nan),
            'time': raw.get('time', np.nan),
        })

    numeric_cols = ['year', 'month', 'day_num', 'sigma', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget', 'loss', 'time']
    for col in numeric_cols:
        mapped[col] = pd.to_numeric(mapped[col], errors='coerce')
    return mapped


def load_all_estimates(files: Dict[str, Path]) -> pd.DataFrame:
    frames = [load_one_estimate_file(name, path) for name, path in files.items()]
    out = pd.concat(frames, ignore_index=True)
    out = out.loc[out['year'].isin(YEARS) & out['month'].eq(MONTH) & out['day_num'].isin(COMMON_DAYS)].copy()
    out = out.sort_values(['year', 'day_num', 'model']).reset_index(drop=True)
    return out


estimates = load_all_estimates(ESTIMATE_FILES)
param_cols = ['year', 'model', 'day', 'sigma', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget', 'loss', 'time']
param_table = estimates[param_cols].round(4)
param_csv = OUTPUT_DIR / 'three_model_parameter_estimates_2022_2025_07.csv'
param_table.to_csv(param_csv, index=False, float_format='%.4f')
print(f'Saved {param_csv}')
display(param_table.head(12))


In [ ]:
DATA_CACHE = {}
GRID_CACHE = {}
RESIDUAL_GRID_CACHE = {}


def load_year_raw_map(year: int) -> Dict[str, pd.DataFrame]:
    if year not in DATA_CACHE:
        path = DATA_ROOT / f'pickle_{year}' / f'tco_grid_{str(year)[2:]}_{MONTH:02d}.pkl'
        with open(path, 'rb') as f:
            DATA_CACHE[year] = pickle.load(f)
        print(f'Loaded {len(DATA_CACHE[year])} hourly maps for {year}-{MONTH:02d}')
    return DATA_CACHE[year]


def build_grid_meta(year: int):
    if year in GRID_CACHE:
        return GRID_CACHE[year]
    raw = load_year_raw_map(year)
    first_key = sorted(raw)[0]
    df = raw[first_key].copy()
    mask = (
        df['Latitude'].between(LAT_RANGE[0], LAT_RANGE[1])
        & df['Longitude'].between(LON_RANGE[0], LON_RANGE[1])
    ).to_numpy()
    sub = df.loc[mask].copy()
    lats = np.sort(sub['Latitude'].round(6).unique())
    lons = np.sort(sub['Longitude'].round(6).unique())
    lat_to_i = {float(v): i for i, v in enumerate(lats)}
    lon_to_j = {float(v): j for j, v in enumerate(lons)}
    lat_idx = sub['Latitude'].round(6).map(lat_to_i).to_numpy(dtype=int)
    lon_idx = sub['Longitude'].round(6).map(lon_to_j).to_numpy(dtype=int)
    meta = {
        'mask': mask,
        'lats': lats,
        'lons': lons,
        'lat_idx': lat_idx,
        'lon_idx': lon_idx,
        'lat_step': float(np.nanmedian(np.diff(lats))),
        'lon_step': float(np.nanmedian(np.diff(lons))),
    }
    GRID_CACHE[year] = meta
    print(f'{year}: grid shape={len(lats)}x{len(lons)}, lat_step={meta["lat_step"]:.4f}, lon_step={meta["lon_step"]:.4f}')
    return meta


def keys_for_day(raw_map: Dict[str, pd.DataFrame], day: int):
    token = f'day{int(day):02d}'
    return [k for k in sorted(raw_map) if token in k]


def compute_day_residual_grids(year: int, day: int):
    cache_key = (year, day)
    if cache_key in RESIDUAL_GRID_CACHE:
        return RESIDUAL_GRID_CACHE[cache_key]

    raw = load_year_raw_map(year)
    meta = build_grid_meta(year)
    day_keys = keys_for_day(raw, day)[:8]
    if len(day_keys) < 8:
        print(f'Warning: {year}-07-{day:02d} has {len(day_keys)} hourly maps')

    y_parts = []
    hour_parts = []
    lat_parts = []
    lon_parts = []
    valid_parts = []

    for hour_idx, key in enumerate(day_keys):
        sub = raw[key].loc[meta['mask'], ['Latitude', 'Longitude', 'ColumnAmountO3']].copy()
        vals = pd.to_numeric(sub['ColumnAmountO3'], errors='coerce').to_numpy(dtype=float)
        lat_vals = pd.to_numeric(sub['Latitude'], errors='coerce').to_numpy(dtype=float)
        lon_vals = pd.to_numeric(sub['Longitude'], errors='coerce').to_numpy(dtype=float)
        valid = np.isfinite(vals) & np.isfinite(lat_vals) & np.isfinite(lon_vals)
        y_parts.append(vals)
        hour_parts.append(np.full(vals.shape, hour_idx, dtype=int))
        lat_parts.append(lat_vals)
        lon_parts.append(lon_vals)
        valid_parts.append(valid)

    y_all = np.concatenate(y_parts)
    hour_all = np.concatenate(hour_parts)
    lat_all = np.concatenate(lat_parts)
    lon_all = np.concatenate(lon_parts)
    valid_all = np.concatenate(valid_parts)

    n_hours = len(day_keys)
    hour_design = np.zeros((valid_all.sum(), n_hours), dtype=float)
    hour_design[np.arange(valid_all.sum()), hour_all[valid_all]] = 1.0
    lat_mean = np.nanmean(lat_all[valid_all])
    lon_mean = np.nanmean(lon_all[valid_all])
    lat_center = lat_all[valid_all] - lat_mean
    lon_center = lon_all[valid_all] - lon_mean
    x = np.column_stack([hour_design, lat_center, lon_center])
    beta, *_ = np.linalg.lstsq(x, y_all[valid_all], rcond=None)

    grids = []
    variances = []
    fitted_means = []
    start = 0
    n_cells = len(meta['lat_idx'])
    for hour_idx, key in enumerate(day_keys):
        stop = start + n_cells
        vals = y_all[start:stop]
        lat_vals = lat_all[start:stop]
        lon_vals = lon_all[start:stop]
        valid = valid_all[start:stop]
        x_hour = np.zeros((n_cells, n_hours + 2), dtype=float)
        x_hour[:, hour_idx] = 1.0
        x_hour[:, n_hours] = lat_vals - lat_mean
        x_hour[:, n_hours + 1] = lon_vals - lon_mean
        mean_hat = x_hour @ beta
        resid = vals - mean_hat
        resid[~valid] = np.nan

        grid = np.full((len(meta['lats']), len(meta['lons'])), np.nan, dtype=float)
        grid[meta['lat_idx'], meta['lon_idx']] = resid
        grids.append(grid)
        variances.append(float(np.nanvar(resid, ddof=1)))
        fitted_means.append(float(np.nanmean(mean_hat[valid])))
        start = stop

    result = {
        'year': year,
        'day': day,
        'hour_keys': day_keys,
        'grids': grids,
        'residual_variances': np.asarray(variances, dtype=float),
        'ols_beta': beta,
        'fitted_hour_mean': np.asarray(fitted_means, dtype=float),
    }
    RESIDUAL_GRID_CACHE[cache_key] = result
    return result


In [ ]:
def shifted_semivariance(grid: np.ndarray, direction: str, offset: int) -> Tuple[float, int]:
    if offset == 0:
        raise ValueError('Zero spatial lag is excluded from pure spatial semivariograms.')
    if direction == 'lat':
        a = grid[:-offset, :]
        b = grid[offset:, :]
    elif direction == 'lon':
        a = grid[:, :-offset]
        b = grid[:, offset:]
    else:
        raise ValueError(direction)
    mask = np.isfinite(a) & np.isfinite(b)
    n = int(mask.sum())
    if n == 0:
        return np.nan, 0
    diff = a[mask] - b[mask]
    return float(0.5 * np.mean(diff * diff)), n


def compute_day_empirical_semivariogram(year: int, day: int):
    meta = build_grid_meta(year)
    residual = compute_day_residual_grids(year, day)
    grids = residual['grids']
    residual_variances = residual['residual_variances']

    lat_offsets = [int(round(float(lag) / meta['lat_step'])) for lag in LAT_LAGS]
    lon_offsets = [int(round(float(lag) / meta['lon_step'])) for lag in LON_LAGS]
    lat_hourly = []
    lon_hourly = []
    lat_counts = []
    lon_counts = []

    for h, grid in enumerate(grids[:8]):
        lat_vals, lat_ns = zip(*[shifted_semivariance(grid, 'lat', off) for off in lat_offsets])
        lon_vals, lon_ns = zip(*[shifted_semivariance(grid, 'lon', off) for off in lon_offsets])
        lat_vals = np.asarray(lat_vals, dtype=float)
        lon_vals = np.asarray(lon_vals, dtype=float)
        if NORMALIZE_EMPIRICAL:
            denom = residual_variances[h]
            if np.isfinite(denom) and denom > 0:
                lat_vals = lat_vals / denom
                lon_vals = lon_vals / denom
            else:
                lat_vals[:] = np.nan
                lon_vals[:] = np.nan
        lat_hourly.append(lat_vals)
        lon_hourly.append(lon_vals)
        lat_counts.append(lat_ns)
        lon_counts.append(lon_ns)

    lat_hourly = np.asarray(lat_hourly, dtype=float)
    lon_hourly = np.asarray(lon_hourly, dtype=float)
    lat_counts = np.asarray(lat_counts, dtype=int)
    lon_counts = np.asarray(lon_counts, dtype=int)

    return {
        'year': year,
        'day': day,
        'hour_keys': residual['hour_keys'],
        'lat_lags': LAT_LAGS,
        'lon_lags': LON_LAGS,
        'lat_hourly': lat_hourly,
        'lon_hourly': lon_hourly,
        'lat_mean': np.nanmean(lat_hourly, axis=0),
        'lon_mean': np.nanmean(lon_hourly, axis=0),
        'lat_counts': lat_counts,
        'lon_counts': lon_counts,
        'residual_variances': residual_variances,
        'mean_model': MEAN_MODEL,
    }


In [ ]:
MODEL_STYLE = {
    'Vecchia_mm20': {'label': 'Vecchia mm20', 'color': '#d62728'},
    'Debiased_Whittle': {'label': 'Debiased Whittle', 'color': '#ff7f0e'},
    'Corridor_4x4_lag643': {'label': 'Corridor 4x4', 'color': '#d4a017'},
}
BLUE_COLORS = plt.cm.Blues(np.linspace(0.35, 0.92, 8))


def fitted_spatial_semivariogram(lags: np.ndarray, row: pd.Series, direction: str) -> np.ndarray:
    sigmasq = float(row['sigma'])
    nugget = float(row['nugget'])
    range_param = float(row['range_lat'] if direction == 'lat' else row['range_lon'])
    h = np.asarray(lags, dtype=float)
    scaled = np.abs(h) / max(range_param, 1e-12)
    if np.isclose(SMOOTH, 0.5):
        corr = np.exp(-scaled)
    elif np.isclose(SMOOTH, 1.5):
        corr = (1.0 + scaled) * np.exp(-scaled)
    else:
        raise ValueError('This notebook currently implements smooth 0.5 or 1.5 for fitted curves.')
    gamma = nugget + sigmasq * (1.0 - corr)
    if NORMALIZE_EMPIRICAL:
        sill = sigmasq + nugget
        gamma = gamma / sill if np.isfinite(sill) and sill > 0 else np.full_like(gamma, np.nan)
    return gamma


def estimate_rows_for_day(year: int, day: int) -> pd.DataFrame:
    return estimates.loc[estimates['year'].eq(year) & estimates['day_num'].eq(day)].copy()


def summarize_curve_error(year: int, day: int, emp: dict):
    rows = []
    est_day = estimate_rows_for_day(year, day)
    for direction, lags, empirical_mean in [
        ('lat', emp['lat_lags'], emp['lat_mean']),
        ('lon', emp['lon_lags'], emp['lon_mean']),
    ]:
        for _, est in est_day.iterrows():
            fitted = fitted_spatial_semivariogram(np.asarray(lags), est, direction)
            valid = np.isfinite(empirical_mean) & np.isfinite(fitted)
            residual = empirical_mean[valid] - fitted[valid]
            rows.append({
                'year': year,
                'month': MONTH,
                'day_num': day,
                'direction': direction,
                'model': est['model'],
                'rmse': float(np.sqrt(np.mean(residual ** 2))) if residual.size else np.nan,
                'mae': float(np.mean(np.abs(residual))) if residual.size else np.nan,
                'bias_emp_minus_fit': float(np.mean(residual)) if residual.size else np.nan,
                'mean_model': MEAN_MODEL,
                'normalized': NORMALIZE_EMPIRICAL,
                'sigma': est['sigma'],
                'range_lat': est['range_lat'],
                'range_lon': est['range_lon'],
                'range_time': est['range_time'],
                'nugget': est['nugget'],
            })
    return rows


In [ ]:
def plot_day_semivariogram(year: int, day: int, emp: dict, out_dir: Path, save=True):
    fig, axes = plt.subplots(1, 2, figsize=(15.8, 6.4), constrained_layout=True)
    est_day = estimate_rows_for_day(year, day)

    panels = [
        ('lat', 'Latitude-direction residual raw semivariogram', emp['lat_lags'], emp['lat_hourly'], emp['lat_mean']),
        ('lon', 'Longitude-direction residual raw semivariogram', emp['lon_lags'], emp['lon_hourly'], emp['lon_mean']),
    ]

    for ax, (direction, title, lags, hourly, daily_mean) in zip(axes, panels):
        for h in range(min(8, hourly.shape[0])):
            ax.plot(lags, hourly[h], color=BLUE_COLORS[h], lw=1.25, alpha=0.86, label=f'Hour {h + 1}')
        ax.plot(lags, daily_mean, color='black', lw=3.1, label='Daily mean over 8 hours', zorder=5)

        first_lag = float(np.nanmin(lags))
        max_lag = float(np.nanmax(lags))
        dense_lags = np.linspace(first_lag, max_lag, 240)
        for _, est in est_day.iterrows():
            style = MODEL_STYLE.get(est['model'], {'label': est['model'], 'color': '0.25'})
            fitted = fitted_spatial_semivariogram(dense_lags, est, direction)
            ax.plot(dense_lags, fitted, color=style['color'], lw=2.4, label=f'{style["label"]} fitted t=0')

        ax.set_ylim(*Y_LIMIT)
        ax.set_xlim(0.0, max_lag)
        ax.set_xlabel(f'{direction} lag (degrees)')
        ax.set_ylabel('residual semivariance')
        ax.set_title(title)
        ax.grid(True, alpha=0.32)

    fig.suptitle(f'{year}-07-{day:02d}: residual raw hourly/daily spatial semivariogram vs fitted t=0 models', fontsize=15)
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=6, fontsize=8.4, frameon=True, bbox_to_anchor=(0.5, -0.070))
    if save:
        out_path = out_dir / f'daily_spatial_semivariogram_residual_raw_three_models_{year}_07_day{day:02d}.png'
        fig.savefig(out_path, bbox_inches='tight')
    return fig


In [ ]:
RUN_ALL = True
SHOW_EACH_DAY = False

all_empirical_rows = []
all_fit_rows = []
all_error_rows = []

if RUN_ALL:
    for year in YEARS:
        year_dir = OUTPUT_DIR / f'year_{year}'
        daily_dir = year_dir / 'daily_semivariograms'
        year_dir.mkdir(parents=True, exist_ok=True)
        daily_dir.mkdir(parents=True, exist_ok=True)
        pdf_path = year_dir / f'daily_spatial_semivariogram_residual_raw_three_models_{year}_07_days01_28.pdf'
        print(f'\n=== Processing {year}-07 ===')
        with PdfPages(pdf_path) as pdf:
            for day in COMMON_DAYS:
                print(f'Processing {year}-07-{day:02d}...')
                emp = compute_day_empirical_semivariogram(year, day)
                for direction, lags, hourly, daily_mean, counts in [
                    ('lat', emp['lat_lags'], emp['lat_hourly'], emp['lat_mean'], emp['lat_counts']),
                    ('lon', emp['lon_lags'], emp['lon_hourly'], emp['lon_mean'], emp['lon_counts']),
                ]:
                    for j, lag in enumerate(lags):
                        row = {
                            'year': year,
                            'month': MONTH,
                            'day_num': day,
                            'direction': direction,
                            'lag': float(lag),
                            'daily_mean': float(daily_mean[j]) if np.isfinite(daily_mean[j]) else np.nan,
                            'mean_pair_count': float(np.nanmean(counts[:, j])),
                            'mean_model': MEAN_MODEL,
                            'normalized': NORMALIZE_EMPIRICAL,
                        }
                        for h in range(hourly.shape[0]):
                            row[f'hour_{h + 1}'] = float(hourly[h, j]) if np.isfinite(hourly[h, j]) else np.nan
                            row[f'n_pairs_hour_{h + 1}'] = int(counts[h, j])
                            row[f'residual_var_hour_{h + 1}'] = float(emp['residual_variances'][h]) if h < len(emp['residual_variances']) else np.nan
                        all_empirical_rows.append(row)

                est_day = estimate_rows_for_day(year, day)
                for direction, lags in [('lat', emp['lat_lags']), ('lon', emp['lon_lags'])]:
                    for _, est in est_day.iterrows():
                        fitted = fitted_spatial_semivariogram(np.asarray(lags), est, direction)
                        for lag, gamma in zip(lags, fitted):
                            all_fit_rows.append({
                                'year': year,
                                'month': MONTH,
                                'day_num': day,
                                'direction': direction,
                                'lag': float(lag),
                                'model': est['model'],
                                'fitted_gamma_t0': float(gamma),
                                'mean_model': MEAN_MODEL,
                                'normalized': NORMALIZE_EMPIRICAL,
                                'sigma': est['sigma'],
                                'range_lat': est['range_lat'],
                                'range_lon': est['range_lon'],
                                'range_time': est['range_time'],
                                'nugget': est['nugget'],
                            })
                all_error_rows.extend(summarize_curve_error(year, day, emp))

                fig = plot_day_semivariogram(year, day, emp, daily_dir, save=True)
                pdf.savefig(fig, bbox_inches='tight')
                if SHOW_EACH_DAY:
                    plt.show()
                else:
                    plt.close(fig)
        print(f'Saved {pdf_path}')

empirical_table = pd.DataFrame(all_empirical_rows).round(4)
fitted_table = pd.DataFrame(all_fit_rows).round(4)
error_table = pd.DataFrame(all_error_rows).round(4)

empirical_csv = OUTPUT_DIR / 'empirical_daily_spatial_semivariogram_residual_raw_2022_2025_07.csv'
fitted_csv = OUTPUT_DIR / 'fitted_t0_spatial_semivariogram_residual_raw_three_models_2022_2025_07.csv'
error_csv = OUTPUT_DIR / 'spatial_semivariogram_residual_raw_model_curve_error_2022_2025_07.csv'
empirical_table.to_csv(empirical_csv, index=False, float_format='%.4f')
fitted_table.to_csv(fitted_csv, index=False, float_format='%.4f')
error_table.to_csv(error_csv, index=False, float_format='%.4f')
print(f'Saved {empirical_csv}')
print(f'Saved {fitted_csv}')
print(f'Saved {error_csv}')

for year in YEARS:
    year_dir = OUTPUT_DIR / f'year_{year}'
    empirical_table.loc[empirical_table['year'].eq(year)].to_csv(year_dir / f'empirical_daily_spatial_semivariogram_residual_raw_{year}_07.csv', index=False, float_format='%.4f')
    fitted_table.loc[fitted_table['year'].eq(year)].to_csv(year_dir / f'fitted_t0_spatial_semivariogram_residual_raw_three_models_{year}_07.csv', index=False, float_format='%.4f')
    error_table.loc[error_table['year'].eq(year)].to_csv(year_dir / f'spatial_semivariogram_residual_raw_model_curve_error_{year}_07.csv', index=False, float_format='%.4f')

display(error_table.head(12))


In [ ]:
# Compact summaries of fitted spatial semivariogram agreement with the empirical daily mean.
summary_by_direction = (
    error_table
    .groupby(['direction', 'model'])
    .agg(
        n=('rmse', 'size'),
        rmse_mean=('rmse', 'mean'),
        rmse_median=('rmse', 'median'),
        mae_mean=('mae', 'mean'),
        mae_median=('mae', 'median'),
        bias_mean=('bias_emp_minus_fit', 'mean'),
        bias_median=('bias_emp_minus_fit', 'median'),
    )
    .round(4)
    .reset_index()
)
summary_overall = (
    error_table
    .groupby('model')
    .agg(
        n=('rmse', 'size'),
        rmse_mean=('rmse', 'mean'),
        rmse_median=('rmse', 'median'),
        mae_mean=('mae', 'mean'),
        mae_median=('mae', 'median'),
        bias_mean=('bias_emp_minus_fit', 'mean'),
        bias_median=('bias_emp_minus_fit', 'median'),
    )
    .round(4)
    .reset_index()
)
win_rows = []
for keys, group in error_table.groupby(['year', 'day_num', 'direction']):
    best_idx = group['rmse'].idxmin()
    win_rows.append({
        'year': keys[0],
        'day_num': keys[1],
        'direction': keys[2],
        'winner': error_table.loc[best_idx, 'model'],
        'winning_rmse': error_table.loc[best_idx, 'rmse'],
    })
win_table = pd.DataFrame(win_rows)
win_counts = (
    win_table
    .groupby(['direction', 'winner'])
    .size()
    .reset_index(name='wins')
    .sort_values(['direction', 'wins'], ascending=[True, False])
)

summary_by_direction_path = OUTPUT_DIR / 'spatial_semivariogram_residual_raw_model_error_summary_by_direction_2022_2025_07.csv'
summary_overall_path = OUTPUT_DIR / 'spatial_semivariogram_residual_raw_model_error_summary_overall_2022_2025_07.csv'
win_counts_path = OUTPUT_DIR / 'spatial_semivariogram_residual_raw_model_win_counts_2022_2025_07.csv'
summary_by_direction.to_csv(summary_by_direction_path, index=False, float_format='%.4f')
summary_overall.to_csv(summary_overall_path, index=False, float_format='%.4f')
win_counts.to_csv(win_counts_path, index=False)
print(f'Saved {summary_by_direction_path}')
print(f'Saved {summary_overall_path}')
print(f'Saved {win_counts_path}')
display(summary_by_direction)
display(summary_overall)
display(win_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), constrained_layout=True)
for ax, direction in zip(axes, ['lat', 'lon']):
    sub = error_table.loc[error_table['direction'].eq(direction)].copy()
    sns.boxplot(data=sub, x='model', y='rmse', hue='model', dodge=False, ax=ax)
    sns.stripplot(data=sub, x='model', y='rmse', color='black', alpha=0.35, size=2.5, ax=ax)
    if ax.legend_:
        ax.legend_.remove()
    ax.set_title(f'{direction} direction: daily mean empirical vs fitted t=0')
    ax.set_xlabel('model')
    ax.set_ylabel('RMSE over positive lags')
    ax.tick_params(axis='x', rotation=20)
summary_plot = OUTPUT_DIR / 'spatial_semivariogram_residual_raw_model_error_boxplot_2022_2025_07.png'
fig.savefig(summary_plot, bbox_inches='tight')
print(f'Saved {summary_plot}')
plt.show()


## Notes

- The empirical curves are computed from residual maps after fitting `ColumnAmountO3 ~ hour fixed effects + latitude + longitude` separately for each day.
- The thick black curve is the pointwise average of the 8 hourly empirical residual semivariograms for that day.
- `NORMALIZE_EMPIRICAL = False`.
- For the normalized notebook, hourly empirical curves are divided by the corresponding hourly residual sample variance; fitted model curves are divided by `sigma + nugget`.
- The fitted curves are `t=0` spatial semivariograms. Therefore the advection estimates do not affect these curves.
- Zero spatial lag is intentionally excluded. The pure spatial semivariogram is plotted from the first grid lag onward: latitude `0.044` and longitude `0.063` degrees, leaving the origin-to-first-lag region blank.
- Numeric CSV outputs are rounded to 4 decimal places.
